<a href="https://colab.research.google.com/github/ishanallasanagala/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [107]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [108]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [109]:
# TODO
print('shape:', df.shape)
print()
print(df.dtypes)
print()
print('nulls per column:')
print(df.isnull().sum())
print()
print('exact duplicate rows:', df.duplicated().sum())

shape: (8, 6)

order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

nulls per column:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

exact duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1. _..._
2. _..._
3. _..._
4. _..._
5. _..._

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [110]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy()  # TODO: df with duplicates dropped, copied


log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [111]:
clean['price'] = clean['price'].str.strip('$').astype(float)

assert clean['price'].dtype == float
log('price', 'stripped string formatting and converted to float', len(clean))

[price] stripped string formatting and converted to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [112]:
# TODO: apply your decision, then log both separately
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum() # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum() # TODO: count of negative quantities

# Decision: Drop missing quantities (cannot assume a sale happened)
# and drop negative quantities (computing GROSS revenue, excluding refunds)
clean = clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()
clean['qty'] = clean['qty'].astype(int)

log('qty', 'dropped rows with missing quantity', missing)
log('qty', 'dropped rows with negative quantity (gross revenue)', negative)

[qty] dropped rows with missing quantity (1 row(s))
[qty] dropped rows with negative quantity (gross revenue) (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [113]:
print('before:', sorted(clean['category'].unique()))

clean['category'] = (
    clean['category'].str.strip()
    .str.lower()
    .str.replace('-', '', regex=False)
)

CATEGORY_MAP = {
    'apparel': 'merch',
    'raingear': 'raingear'
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))

log('category', f"normalized text and mapped variants (started with {df['category'].nunique()} distinct, ended with {clean['category'].nunique()})", len(clean))

before: ['Apparel', 'Food', 'Merch', 'food', 'rain-gear']
after:  ['food', 'merch', 'raingear']
[category] normalized text and mapped variants (started with 6 distinct, ended with 3) (5 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [114]:
print('before:', sorted(clean['item'].dropna().unique()))

clean['item'] = clean['item'].str.strip().str.title()
ITEM_MAP = {'Cheese Burger': 'Cheeseburger'}
clean['item'] = clean['item'].replace(ITEM_MAP)

# Decision: Fill missing item names with 'Unknown Item' to keep the revenue
missing_items = clean['item'].isna().sum()
clean['item'] = clean['item'].fillna('Unknown Item')

print('after: ', sorted(clean['item'].unique()))
log('item', f"standardized casing, mapped variants, filled {missing_items} missing item(s) with 'Unknown Item'", len(clean))

before: ['Cheeseburger', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after:  ['Cheeseburger', 'Rain Poncho', 'Unknown Item', 'Uva T-Shirt']
[item] standardized casing, mapped variants, filled 1 missing item(s) with 'Unknown Item' (5 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [115]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')
failed_ts = clean['ts'].isna().sum()

# Decision: Keep the NaT rows so we don't lose revenue, but note the failure
clean['hour'] = clean['ts'].dt.hour

log('ts', f"parsed to datetime and extracted hour ({failed_ts} unparseable coerced to NaT)", len(clean))

[ts] parsed to datetime and extracted hour (1 unparseable coerced to NaT) (5 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [116]:
# 1. No exact duplicates remain
assert clean.duplicated().sum() == 0, 'duplicates remain'
# 2. Price is numeric
assert clean['price'].dtype == float, 'price is not numeric'
# 3. Quantity is strictly positive
assert clean['qty'].min() >= 1, 'non-positive quantities remain'
# 4. Item has no nulls
assert clean['item'].isna().sum() == 0, 'missing items remain'
# 5. Categories are all lowercase
assert clean['category'].str.islower().all(), 'inconsistent category text'
# 6. Timestamps are genuine datetime objects
assert pd.api.types.is_datetime64_any_dtype(clean['ts']), 'ts is not a datetime'

clean['revenue'] = clean['qty'] * clean['price']

print(f"Rows: {len(clean)}")
print(f"Units: {clean['qty'].sum()}")
print(f"Revenue: ${clean['revenue'].sum():.2f}")
print(f"Categories: {clean['category'].nunique()}")

Rows: 5
Units: 10
Revenue: $106.50
Categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [117]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,stripped string formatting and converted to float,7
2,qty,dropped rows with missing quantity,1
3,qty,dropped rows with negative quantity (gross rev...,1
4,category,normalized text and mapped variants (started w...,5
5,item,"standardized casing, mapped variants, filled 1...",5
6,ts,parsed to datetime and extracted hour (1 unpar...,5


**The decision that mattered most:** _..._

**Revenue with it:** _..._  **Revenue without it:** _..._

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work the last few minutes in groups of 4–5, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Wednesday 11:59pm ET**. One submission per person, not per group.

In [118]:
# Checkpoint
rows_after = len(clean)
revenue_after = round(clean['revenue'].sum(), 2)
biggest_decision = 'Dropping negative quantity rows instead of keeping them'
revenue_other_way = revenue_after - (abs(df[pd.to_numeric(df['qty'], errors='coerce') < 0]['qty'].astype(float).sum()) * df[pd.to_numeric(df['qty'], errors='coerce') < 0]['price'].astype(str).str.replace('$', '', regex=False).astype(float).sum()) # Or manually plug in the Net Revenue number from your specific data run

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 106.5
decision that mattered: Dropping negative quantity rows instead of keeping them
revenue the other way: 88.5
